In [1]:
!pip install pandas numpy matplotlib seaborn scikit-learn xgboost shap openpyxl

#Cell 1 — Install & imports

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.float_format", "{:.2f}".format)
pd.set_option("display.max_columns", 20)

print("All libraries loaded ✓")

All libraries loaded ✓


#Cell 2-Load all three files

In [7]:
# Update paths if your files are in a subfolder
daily   = pd.read_csv("C:/Users/Hello User/Downloads/salesdaily.csv")
weekly  = pd.read_csv("C:/Users/Hello User/Downloads/salesweekly.csv")
monthly = pd.read_csv("C:/Users/Hello User/Downloads/salesmonthly.csv")

# The date column is called 'datum' in this dataset
for name, df in [("Daily", daily), ("Weekly", weekly), ("Monthly", monthly)]:
    print(f"\n{'='*40}")
    print(f"  {name} dataset")
    print(f"  Shape     : {df.shape}")
    print(f"  Columns   : {df.columns.tolist()}")
    print(f"  Date range: {df['datum'].min()} → {df['datum'].max()}")


  Daily dataset
  Shape     : (2106, 13)
  Columns   : ['datum', 'M01AB', 'M01AE', 'N02BA', 'N02BE', 'N05B', 'N05C', 'R03', 'R06', 'Year', 'Month', 'Hour', 'Weekday Name']
  Date range: 1/1/2015 → 9/9/2019

  Weekly dataset
  Shape     : (302, 9)
  Columns   : ['datum', 'M01AB', 'M01AE', 'N02BA', 'N02BE', 'N05B', 'N05C', 'R03', 'R06']
  Date range: 1/1/2017 → 9/9/2018

  Monthly dataset
  Shape     : (70, 9)
  Columns   : ['datum', 'M01AB', 'M01AE', 'N02BA', 'N02BE', 'N05B', 'N05C', 'R03', 'R06']
  Date range: 2014-01-31 → 2019-10-31


In [2]:
#Cell 3-Fix date columns & inspect dtypes

In [9]:
# Parse dates
daily["datum"]   = pd.to_datetime(daily["datum"])
weekly["datum"]  = pd.to_datetime(weekly["datum"])
monthly["datum"] = pd.to_datetime(monthly["datum"])

# Drug columns (all except datum and any hour/weekday cols)
DRUG_COLS = ["M01AB", "M01AE", "N02BA", "N02BE", "N05B", "N05C", "R03", "R06"]

print("Daily dtypes:")
print(daily.dtypes)

print("\nMissing values — Monthly:")
print(monthly[DRUG_COLS].isnull().sum())

print("\nBasic stats — Monthly sales:")
print(monthly[DRUG_COLS].describe().round(2))

Daily dtypes:
datum           datetime64[ns]
M01AB                  float64
M01AE                  float64
N02BA                  float64
N02BE                  float64
N05B                   float64
N05C                   float64
R03                    float64
R06                    float64
Year                     int64
Month                    int64
Hour                     int64
Weekday Name            object
dtype: object

Missing values — Monthly:
M01AB    0
M01AE    0
N02BA    0
N02BE    0
N05B     0
N05C     0
R03      0
R06      0
dtype: int64

Basic stats — Monthly sales:
       M01AB  M01AE  N02BA   N02BE   N05B  N05C    R03    R06
count  70.00  70.00  70.00   70.00  70.00 70.00  70.00  70.00
mean  149.99 116.51 115.02  892.54 262.12 17.84 167.68  86.66
std    31.49  27.89  31.25  338.84  85.06  8.48  81.77  45.86
min     0.00   0.00   0.00    0.00   1.00  0.00   0.00   0.00
25%   137.49 103.52  94.38  648.19 223.75 12.00 112.00  49.88
50%   154.64 114.84 117.22  865.82 250.

In [8]:
#Cell 4 — Check for extra columns in daily file

In [11]:
# Daily file often has extra columns: Year, Month, Hour, Day of Week
print("Daily extra columns:", [c for c in daily.columns if c not in DRUG_COLS + ["datum"]])

# Standardise: keep only datum + drug cols across all files
daily_clean   = daily[["datum"] + DRUG_COLS].copy()
weekly_clean  = weekly[["datum"] + DRUG_COLS].copy()
monthly_clean = monthly[["datum"] + DRUG_COLS].copy()

# Add a 'granularity' label (useful for merging later)
daily_clean["granularity"]   = "daily"
weekly_clean["granularity"]  = "weekly"
monthly_clean["granularity"] = "monthly"

print("\nClean shapes:")
print(f"  Daily  : {daily_clean.shape}")
print(f"  Weekly : {weekly_clean.shape}")
print(f"  Monthly: {monthly_clean.shape}")

Daily extra columns: ['Year', 'Month', 'Hour', 'Weekday Name']

Clean shapes:
  Daily  : (2106, 10)
  Weekly : (302, 10)
  Monthly: (70, 10)


In [10]:
#Cell 5 — Quick sanity check: do daily totals match monthly?

In [13]:
# Resample daily to monthly and compare with monthly file
daily_clean["year_month"] = daily_clean["datum"].dt.to_period("M")
daily_resampled = daily_clean.groupby("year_month")[DRUG_COLS].sum()

monthly_clean["year_month"] = monthly_clean["datum"].dt.to_period("M")
monthly_indexed = monthly_clean.set_index("year_month")[DRUG_COLS]

# Correlation between resampled daily and provided monthly
corr = daily_resampled.corrwith(monthly_indexed).round(4)
print("Correlation (resampled daily vs monthly file):")
print(corr)
# Should be very high (>0.95) — confirms data consistency

Correlation (resampled daily vs monthly file):
M01AB   0.70
M01AE   0.66
N02BA   0.67
N02BE   0.82
N05B    0.79
N05C    0.81
R03     0.95
R06     0.98
dtype: float64
